<a href="https://colab.research.google.com/github/guangyoung/tradeSimulation/blob/main/quantgenius_tradingSimulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

<p><font size="6">STOCK TRADING SIMULATION</font></p>
<p><font size="4">Using Real-time Rest API Based Trade Signals From</font></p>

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQwU1UsIOz6KlmYSN-63rR7dZrMVl63IGpgcA&s" width="150" alt="QuantGenius Logo">

<p><font size="6">QUANTGENIUS</font></p>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/username/repo/blob/main/notebook.ipynb)
&nbsp; ![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)
&nbsp; ![Python: 3.10+](https://img.shields.io/badge/Python-3.10+-blue.svg)

</div>

---

### **Overview**
Simulasi ini dirancang untuk menguji efektivitas sinyal AI dari **QuantGenius** menggunakan data portofolio real-time. Melalui notebook ini, Anda dapat:
1. **Mengunggah** dataset harga saham kustom.
2. **Mengintegrasikan** API Key QuantGenius.
3. **Menganalisis** performa strategi secara institusional.

## **1. Environment Setup**
Menginstal dependensi yang diperlukan dan memuat library standar untuk analisis data serta komunikasi REST API.

In [ ]:
# @title Install & Import Dependencies { display-mode: "form" }

# 1. Install libraries yang mungkin belum ada di environment Colab standar
!pip install -q pandas numpy requests matplotlib seaborn

# 2. Import core libraries
import pandas as pd
import numpy as np
import requests
import json
import time
from datetime import datetime

# 3. Visualisasi & Estetika (Standar Quant)
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi tampilan grafik agar profesional (Dark Mode Friendly)
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

print(f"✅ Environment ready at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📦 Pandas version: {pd.__version__}")

✅ Environment ready at: 2026-05-02 14:44:13
📦 Pandas version: 2.2.2


## **2. Simulation Settings & Authentication**
Konfigurasikan parameter simulasi dan hubungkan dengan API QuantGenius untuk menerima sinyal trading.

In [ ]:
# @title Konfigurasi QuantGenius { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. CSS untuk styling agar teks deskripsi rapi
style = {'description_width': 'initial'}
layout_input = widgets.Layout(width='100px')

# --- BAGIAN KIRI ---
api_key = widgets.Password(placeholder='Input Your QuantGenius API-Key', layout=widgets.Layout(width='275px'))
desc_api = widgets.HTML(value="""<div style="font-size: 12px; color: gray;">If you don't have a QuantGenius API-Key. <a href="https://api.quantgenius.ai" target="_blank" style="color: #1a73e8; text-decoration: none;">Click here<a></div>""")

# Row 1 Kiri
init_equity = widgets.FloatText(value=1000000, description='Initial Equity', style=style, layout=widgets.Layout(width='200px'))
comm = widgets.FloatText(value=0.001, description='Commission/trade', style=style, layout=widgets.Layout(width='200px'))
spread = widgets.FloatText(value=0.001, description='Spread/trade', style=style, layout=widgets.Layout(width='200px'))
interest = widgets.FloatText(value=0.03, description='Interest Rate/year', style=style, layout=widgets.Layout(width='200px'))
min_slippage = widgets.FloatText(value=0.03, description='Min Slippage', style=style, layout=widgets.Layout(width='200px'))
max_slippage = widgets.FloatText(value=0.03, description='Max Slippage', style=style, layout=widgets.Layout(width='200px'))
min_execution = widgets.FloatText(value=0.03, description='Min Execution', style=style, layout=widgets.Layout(width='200px'))
max_execution = widgets.FloatText(value=0.03, description='Max Execution', style=style, layout=widgets.Layout(width='200px'))

# Row 2 Kiri (Margin Management)
# mr1 = widgets.FloatText(value=0.5, description='Margin Requirement', style=style, layout=layout_input)
# mm1 = widgets.FloatText(value=0.25, description='Margin Maintenance', style=style, layout=layout_input)
# mr2 = widgets.FloatText(value=0.5, description='Margin Requirement', style=style, layout=layout_input)
# mm2 = widgets.FloatText(value=0.25, description='Margin Maintenance', style=style, layout=layout_input)

left_box = widgets.VBox([
    widgets.HBox([widgets.Label("QuantGenius API Key", layout=widgets.Layout(width='135px')), api_key], layout=widgets.Layout(width='410px')), desc_api,
    widgets.HBox([init_equity, comm]),
    widgets.HBox([spread, interest]),
    widgets.HBox([min_slippage, max_slippage]),
    widgets.HBox([min_execution, max_execution]),
    # widgets.HBox([mr1, mm1, mr2, mm2]),
], layout=widgets.Layout(width='37%'))

# --- BAGIAN KANAN ---
# Membuat deretan Stock (MR & MM)
stocks_mr = [widgets.FloatText(value=0.5, description=f'Stock {i+1}', style=style, layout=widgets.Layout(width='110px')) for i in range(30)]
stocks_mm = [widgets.FloatText(value=0.25, description=f'Stock {i+1}', style=style, layout=widgets.Layout(width='110px')) for i in range(30)]
# --- 1. SELEKTOR MODE ---
mode_selector = widgets.Dropdown(
    options=[('Sama untuk Semua Saham', 'all'), ('Isi Per Saham', 'individual')],
    value='all',
    description='Select margin mode input:',
    style=style,
    layout=widgets.Layout(width='350px')
)

stock_mr_all = widgets.HBox(stocks_mr, layout=widgets.Layout(overflow='auto'))
stock_mm_all = widgets.HBox(stocks_mm, layout=widgets.Layout(overflow='auto'))
stock_mr_global = widgets.FloatText(value=0.5, description='All Stocks', layout=widgets.Layout(width='150px'))
stock_mm_global = widgets.FloatText(value=0.25, description='All Stocks', layout=widgets.Layout(width='150px'))
label_mr = widgets.Label("Margin Requirement :", layout=widgets.Layout(width='750px'))
label_mm = widgets.Label("Maintenance Margin :", layout=widgets.Layout(width='750px'))
label_mr1 = widgets.Label("Margin Requirement :", layout=widgets.Layout(width='125px'))
label_mm1 = widgets.Label("Maintenance Margin :", layout=widgets.Layout(width='125px'))
# desc_right = widgets.HTML("<small style='color:gray;'>QuantGenius telah dilatih dan di uji pada margin trading dengan leverage max hanya 2x, menggunakan 30 saham portofolio...</small>")
tampilan_all = widgets.VBox([widgets.HBox([label_mr,stock_mr_all]),widgets.HBox([label_mm,stock_mm_all])], layout=widgets.Layout(display='none'))
tampilan_global = widgets.VBox([widgets.HBox([label_mr1,stock_mr_global]),widgets.HBox([label_mm1,stock_mm_global])])

# --- 4. LOGIKA INTERAKTIF ---
def on_mode_change(change):
    if change['new'] == 'all':
        tampilan_global.layout.display = 'flex'
        tampilan_all.layout.display = 'none'
    else:
        tampilan_global.layout.display = 'none'
        tampilan_all.layout.display = 'flex'

mode_selector.observe(on_mode_change, names='value')

right_box = widgets.VBox([
    widgets.HBox([mode_selector]),
    tampilan_global,
    tampilan_all
], layout=widgets.Layout(width='61%'))

# --- FOOTER ---
# footer = widgets.HTML("<hr><p style='font-size:11px; color:gray;'><b>PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA. ANDA JUGA BISA MENYESUAIKAN SETTING UNTUK UJI STRESS UNTUK MELIHAT KETANGGUHAN SISTEM. PASTIKAN SETTING ANDA SENYATA MUNGKIN AGAR HASIL SIMULASI BENAR BENAR AKURAT SESUAI TRADING NYATA.</b></p>")
footer = widgets.HTML(
    value="""
    <div style="line-height: 1.5; margin: 0; padding: 0; font-size: 12px; color: gray;">
        <hr>Pastikan setting anda senyata mungkin agar hasil simulasi benar-benar akurat sesuai trading nyata. Anda juga bisa menyesuaikan setting untuk uji stress untuk melihat ketangguhan sistem. >Pastikan setting anda senyata mungkin agar hasil simulasi benar-benar akurat sesuai trading nyata. Anda juga bisa menyesuaikan setting untuk uji stress untuk melihat ketangguhan sistem.
    </div>
    """
)
# --- DISPLAY AKHIR ---
main_container = widgets.HBox([left_box, right_box])

display(main_container, footer)

HTML(value='\n    <div style="line-height: 1.5; margin: 0; padding: 0; font-size: 12px; color: gray;">\n      …

## **3. Dataset Preparation**
Unggah data historis harga saham Anda (CSV) atau gunakan generator data sampel untuk menguji konektivitas API.

In [ ]:
# @title Simulasi Dataset { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display

style = {'description_width': 'initial'}

# --- 1. Radio Buttons (Select Dataset) ---
dataset_radio = widgets.RadioButtons(
    options=['Montecarlo Simulation', 'Local Data'],
    value='Montecarlo Simulation',
    layout=widgets.Layout(width='auto')
)

dataset_box = widgets.HBox([
    widgets.Label('Select Dataset', layout=widgets.Layout(margin='0 15px 0 0')),
    dataset_radio
])

# --- 2. Dropdown (Select Simulation Method) ---
method_dropdown = widgets.Dropdown(
    options=['Method1', 'Method2', 'Method3'],
    value='Method1',
    layout=widgets.Layout(width='250px', margin='5px 0 15px 0')
)

method_label = widgets.Label('Select Simulation Method')

# --- 4. Buttons (Create & Reset) ---
# Menggunakan button_style untuk warna standar atau style.button_color untuk kustom
btn_create = widgets.Button(
    description='Create Dataset',
    layout=widgets.Layout(width='120px', height='30px', margin='0 10px 0 0')
)
btn_create.style.button_color = '#5bc0de' # Biru muda sesuai gambar
btn_create.style.font_weight = 'bold'

btn_reset = widgets.Button(
    description='Reset Dataset',
    layout=widgets.Layout(width='120px', height='30px', margin='0 10px 0 0')
)
btn_reset.style.button_color = '#d9534f' # Merah sesuai gambar
btn_reset.style.font_weight = 'bold'

setting11 = widgets.FloatText(value=0.03, description='Setting11', style=style, layout=widgets.Layout(width='200px'))
setting12 = widgets.FloatText(value=0.03, description='Setting12', style=style, layout=widgets.Layout(width='200px'))
setting13 = widgets.FloatText(value=0.03, description='Setting13', style=style, layout=widgets.Layout(width='200px'))
setting21 = widgets.FloatText(value=0.03, description='Setting21', style=style, layout=widgets.Layout(width='200px'))
setting22 = widgets.FloatText(value=0.03, description='Setting22', style=style, layout=widgets.Layout(width='200px'))
setting23 = widgets.FloatText(value=0.03, description='Setting23', style=style, layout=widgets.Layout(width='200px'))
setting31 = widgets.FloatText(value=0.03, description='Setting31', style=style, layout=widgets.Layout(width='200px'))
setting32 = widgets.FloatText(value=0.03, description='Setting32', style=style, layout=widgets.Layout(width='200px'))
setting33 = widgets.FloatText(value=0.03, description='Setting33', style=style, layout=widgets.Layout(width='200px'))

tampilan_setting1 = widgets.VBox([setting11,setting12,setting13])
tampilan_setting2 = widgets.VBox([setting21,setting22,setting23], layout=widgets.Layout(display='none'))
tampilan_setting3 = widgets.VBox([setting31,setting32,setting33], layout=widgets.Layout(display='none'))

# Logika untuk mengubah label setting secara dinamis
def on_method_change(change):
    if change['new'] == 'Method1':
        tampilan_setting1.layout.display = 'flex'
        tampilan_setting2.layout.display = 'none'
        tampilan_setting3.layout.display = 'none'
    elif change['new'] == 'Method2':
        tampilan_setting1.layout.display = 'none'
        tampilan_setting2.layout.display = 'flex'
        tampilan_setting3.layout.display = 'none'
    else:
        tampilan_setting1.layout.display = 'none'
        tampilan_setting2.layout.display = 'none'
        tampilan_setting3.layout.display = 'flex'


method_dropdown.observe(on_method_change, names='value')

# --- 5. Layouting ---
button_box = widgets.HBox([btn_create, btn_reset],layout=widgets.Layout(margin='15px 0 0 0'))

ui = widgets.VBox([
    dataset_box,
    method_label,
    method_dropdown,
    tampilan_setting1,
    tampilan_setting2,
    tampilan_setting3,
    button_box
], layout=widgets.Layout(padding='10px'))

display(ui)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import io

# Konfigurasi & State
MAX_FILES, uploaded_files = 30, {}
output = widgets.Output()
uploader_container = widgets.HBox()

def create_uploader():
    btn = widgets.FileUpload(accept='.csv,.txt', multiple=True, description='Upload Files')
    btn.observe(on_upload_change, names='value')
    return btn

def on_upload_change(change):
    if not change['new']: return
    # Standarisasi input (tuple/dict) ke list
    new_data = change['new']
    files = new_data if isinstance(new_data, tuple) else [{'name':k, 'content':v['content']} for k,v in new_data.items()]

    with output:
        clear_output()
        added, skipped = 0, []
        for f in files:
            name = f['name']
            if name in uploaded_files: skipped.append(name)
            elif len(uploaded_files) < MAX_FILES:
                uploaded_files[name] = f['content']
                added += 1
            else: print(f"❌ Limit {MAX_FILES} tercapai. '{name}' diabaikan.")

        if skipped: print(f"⚠️ Duplikat dilewati: {', '.join(skipped)}")
        print(f"📊 Status: {len(uploaded_files)}/{MAX_FILES} file. (Berhasil tambah {added})")
        uploader._counter = len(uploaded_files)

def on_process_clicked(b):
    with output:
        clear_output()
        if len(uploaded_files) != 30: return print("❌ Error: Tidak cukup 30 files."f"📊 Status: {len(uploaded_files)}/{MAX_FILES} file.")

        port_ticker, port_data = [], []
        # print(f"⚙️ Memproses {len(uploaded_files)} file...")

        for name, content in uploaded_files.items():
            try:
                df = pd.read_csv(io.BytesIO(content))
                # Auto-detect kolom tanggal
                date_col = next((c for c in df.columns if c.lower() in ['date', 'timestamp', 'datetime']), None)
                if date_col:
                    df[date_col] = pd.to_datetime(df[date_col])
                    df.set_index(date_col, inplace=True)

                df.index = pd.to_datetime(df.index).date
                df = df[pd.to_datetime(df.index).dayofweek < 5].dropna(how='all') # Filter weekday & Clean

                if len(df) > 100:
                    port_ticker.append(name.split('.')[0].upper())
                    port_data.append(df)
            except Exception as e: print(f"❌ Gagal {name}: {e}")

        if port_data:
            # Sync dates & combine
            start, end = max(d.index.min() for d in port_data), min(d.index.max() for d in port_data)
            common_dates = pd.date_range(start, end, freq='B').date

            combined = pd.DataFrame(index=common_dates)
            for t, d in zip(port_ticker, port_data):
                combined[t] = d['Close'].reindex(common_dates)

            combined = combined.ffill().bfill().dropna()

            # --- TAMPILAN STATISTIK HORIZONTAL ---
            print("📊 PORTFOLIO SUMMARY")

            # Membuat dictionary horizontal
            avg_corr = combined.pct_change().corr().mean().mean()
            stats_dict = {
                "Total Tickers": [f"{len(combined.columns)} Assets"],
                "Total Rows": [f"{len(combined)} Days"],
                "Start Date": [str(combined.index.min())],
                "End Date": [str(combined.index.max())],
                "Avg Correlation": [f"{avg_corr:.2f}"]
            }

            # Tampilkan sebagai DataFrame horizontal
            stats_df = pd.DataFrame(stats_dict)
            styled_stats = stats_df.style.hide(axis='index').set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#f1f3f4'), ('color', '#3c4043'), ('text-align', 'center')]}
            ]).set_properties(**{'text-align': 'center', 'border': '1px solid #ddd', 'min-width': '120px'})

            display(styled_stats)
            print("-" * 150)

            # Display Dataframe Head & Tail (Data Harga)
            sep = pd.DataFrame({c: ['...'] for c in combined.columns}, index=['...'])
            styled_data = pd.concat([combined.head(5), sep, combined.tail(5)]).style.set_table_styles([
                {'selector': '.row_heading, th.blank', 'props': [('min-width', '100px')]}
            ])
            display(styled_data)

def on_reset_clicked(b):
    global uploader
    uploaded_files.clear()
    uploader = create_uploader()
    uploader_container.children = [uploader, process_button, reset_button]
    with output:
        clear_output()
        print("🔄 Berhasil direset.")

# Init UI
uploader = create_uploader()
process_button = widgets.Button(description='Process Data', button_style='success', icon='check')
process_button.on_click(on_process_clicked)
reset_button = widgets.Button(description='Reset All', button_style='warning')
reset_button.on_click(on_reset_clicked)

uploader_container.children = [uploader, process_button, reset_button]
print("# QuantGenius File Management")
display(widgets.VBox([uploader_container, output]))

# QuantGenius File Management


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Inisialisasi Output Area
out_sim = widgets.Output()

# 2. Membuat Widget Parameter
style = {'description_width': 'initial'}

# Jumlah saham dikunci ke 30 dan di-disable
w_n_saham = widgets.IntText(value=30, description='Jumlah Saham:', disabled=True, style=style)

w_n_hari = widgets.IntText(value=252, min=20, max=7500, description='Hari Bursa:', style=style)
w_mu = widgets.FloatText(value=0.12, min=-0.5, max=0.5, step=0.01, description='Drift (Mu):', style=style)
w_sigma = widgets.FloatText(value=0.25, min=0.01, max=1.0, step=0.01, description='Volatility (Sigma):', style=style)

# Mengubah rentang harga menjadi satu nilai tetap untuk semua saham
w_s0_fixed = widgets.FloatText(value=100.0, description='Harga Awal (S0):', style=style)

btn_run = widgets.Button(description="Run Simulation", button_style='success', icon='play')
btn_reset_sim = widgets.Button(description="Reset Params", button_style='warning', icon='refresh')

params_box = widgets.VBox([
    widgets.HTML("<h3>⚙️ Monte Carlo Simulation Panel (Fixed Assets)</h3>"),
    w_n_saham, w_n_hari, w_mu, w_sigma, w_s0_fixed,
    widgets.HBox([btn_run, btn_reset_sim])
])

# 3. Fungsi Utama Simulasi
def run_gbm_simulation(b):
    with out_sim:
        clear_output(wait=True)

        # --- LOGIKA SIMULASI GBM ---
        n_saham = 30 # Fixed
        n_hari = w_n_hari.value
        mu, sigma = w_mu.value, w_sigma.value
        s0_val = w_s0_fixed.value
        dt = 1/252

        # Tanpa seed agar pergerakan antar saham tetap berbeda (random shock)
        dates = pd.date_range(start='2024-01-01', periods=n_hari, freq='B').date
        tickers = [f"SIM_{i+1:02d}" for i in range(n_saham)]

        # Semua saham mulai dari harga yang sama
        S0 = np.full(n_saham, s0_val)

        # Kejutan (shock) tetap random untuk tiap saham agar grafik tidak menumpuk jadi 1 garis
        Z = np.random.standard_normal((n_hari, n_saham))

        returns = np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
        path = np.ones((n_hari, n_saham))
        path[0] = S0
        for t in range(1, n_hari):
            path[t] = path[t-1] * returns[t]

        df_sim = pd.DataFrame(path, index=dates, columns=tickers)
        df_sim.index.name = 'Date'

        # --- TAB 1: GRAFIK ---
        out_chart = widgets.Output()
        with out_chart:
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(df_sim, alpha=0.6, linewidth=1)
            ax.set_title(f"Price Paths: 30 Stocks Starting at {s0_val}", fontweight='bold')
            ax.set_ylabel("Price")
            ax.set_xlabel("Date")
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        # --- TAB 2: TABEL DATA ---
        out_table = widgets.Output()
        with out_table:
            avg_corr = df_sim.pct_change().corr().mean().mean()
            stats_df = pd.DataFrame({
                "Fixed Assets": [f"{n_saham}"],
                "Start Price": [f"{s0_val}"],
                "Days": [f"{n_hari}"],
                "Avg Corr": [f"{avg_corr:.2f}"]
            })
            display(widgets.HTML("<b>📊 Statistics Summary</b>"))
            display(stats_df.style.hide(axis='index').set_properties(**{'text-align': 'center', 'border': '1px solid #ddd'}))

            display(widgets.HTML("<br><b>📄 Price Data Preview</b>"))
            sep = pd.DataFrame({c: ['...'] for c in df_sim.columns}, index=['...'])
            preview = pd.concat([df_sim.head(5), sep, df_sim.tail(5)])
            display(preview.style.format(lambda x: f"{x:,.2f}" if isinstance(x, (float, int)) else x))

        tabs = widgets.Tab(children=[out_chart, out_table])
        tabs.set_title(0, '📈 Simulation Chart')
        tabs.set_title(1, '📋 Data Table')
        display(tabs)

def reset_simulation_params(b):
    w_mu.value, w_sigma.value, w_s0_fixed.value = 0.12, 0.25, 100.0
    with out_sim: clear_output()

btn_run.on_click(run_gbm_simulation)
btn_reset_sim.on_click(reset_simulation_params)

# Tampilkan Dashboard
display(widgets.VBox([params_box, out_sim]))

In [2]:
# @title Simulasi Dataset { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# 1. Inisialisasi State & Output Global
data_store = {'uploaded_files': {}} # Menggunakan dict agar mutable di dalam fungsi
out_main = widgets.Output()
out_result = widgets.Output()

test_data = pd.DataFrame()
tanggal = np.array([])
dataHarga = np.array([])

# 2. Widget Pemilih Sumber Data
datasource_radio = widgets.RadioButtons(
    options=[('Upload 30 CSV Files', 'csv'), ('Montecarlo Simulation', 'mc')],
    description='<b>Select Data Source:</b>',
    style={'description_width': 'initial'}
)

toggle_select = widgets.ToggleButtons(
    options=['Upload 30 CSV Files', 'Montecarlo Simulation'],
    # description='Speed:',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    # tooltips=['Description of slow', 'Description of regular', 'Description of fast'],
#     icons=['check'] * 3
)

toggle_button = widgets.HBox([widgets.Label(value="Select Data Source:"), toggle_select])

# --- PANEL A: CSV MANAGEMENT ---
def create_csv_panel():
    MAX_FILES, uploaded_files = 30, {}
    uploader_container = widgets.HBox()
    uploaded_files.clear()

    # Simpan referensi uploader agar bisa diakses handler
    state = {'uploader': None}

    def create_uploader():
        btn = widgets.FileUpload(accept='.csv,.txt', multiple=True, description='Upload Files')
        btn.observe(on_upload_change, names='value')
        return btn

    def on_upload_change(change):
        if not change['new']: return
        # Standarisasi input (tuple/dict) ke list
        new_data = change['new']
        files = new_data if isinstance(new_data, tuple) else [{'name':k, 'content':v['content']} for k,v in new_data.items()]

        with out_result:
            clear_output()
            added, skipped = 0, []
            for f in files:
                name = f['name']
                if name in uploaded_files: skipped.append(name)
                elif len(uploaded_files) < MAX_FILES:
                    uploaded_files[name] = f['content']
                    added += 1
                else: print(f"❌ Limit {MAX_FILES} tercapai. '{name}' diabaikan.")

            if skipped: print(f"⚠️ Duplikat dilewati: {', '.join(skipped)}")
            print(f"📊 Status: {len(uploaded_files)}/{MAX_FILES} file. (Berhasil tambah {added})")
            uploader._counter = len(uploaded_files)

    def on_process_clicked(b):
        global dataHarga, test_data, tanggal
        with out_result:
            clear_output()
            if len(uploaded_files) != 30:
                return print(f"❌ Error: Butuh 30 files, baru ada {len(uploaded_files)}.")

            port_ticker, port_data = [], []

            for name, content in uploaded_files.items():
                try:
                    # 1. Baca data (txt atau csv)
                    # Menggunakan sep=',' karena contoh data Anda dipisah koma
                    df = pd.read_csv(io.BytesIO(content), sep=',')

                    # 2. Tangani kolom <DATE>
                    # Kita cari kolom yang mengandung kata 'DATE' untuk antisipasi perbedaan nama
                    date_col = next((c for c in df.columns if 'DATE' in c.upper()), None)
                    close_col = next((c for c in df.columns if 'CLOSE' in c.upper()), None)

                    if date_col and close_col:
                        # KONVERSI TANGGAL: format '20100409' -> %Y%m%d
                        df[date_col] = pd.to_datetime(df[date_col].astype(str), format='%Y%m%d')
                        df.set_index(date_col, inplace=True)

                        # Pastikan index diurutkan (penting untuk data time-series)
                        df = df.sort_index()

                        # Filter hari kerja & hapus kosong
                        df = df[df.index.dayofweek < 5]

                        # Ambil hanya kolom CLOSE
                        clean_df = df[[close_col]].copy()
                        clean_df.columns = ['Close'] # Standarisasi nama kolom

                        if len(clean_df) > 50: # Minimal data
                            port_ticker.append(name.split('.')[0].upper())
                            port_data.append(clean_df)
                    else:
                        print(f"⚠️ Kolom <DATE> atau <CLOSE> tidak ditemukan di {name}")

                except Exception as e:
                    print(f"❌ Gagal memproses {name}: {e}")

            if port_data:
                # 3. Sinkronisasi Tanggal antar 30 file
                start = max(d.index.min() for d in port_data)
                end = min(d.index.max() for d in port_data)
                common_dates = pd.date_range(start, end, freq='B')

                combined = pd.DataFrame(index=common_dates)
                for t, d in zip(port_ticker, port_data):
                    # Ambil data Close dan reindex ke tanggal yang sama
                    combined[t] = d['Close'].reindex(common_dates)

                # Fill missing data (jika ada hari libur/gap)
                combined = combined.ffill().bfill().dropna()
                combined.index.name = 'Date'

                # --- ASSIGN GLOBAL ---
                test_data = combined
                dataHarga = test_data.to_numpy()
                tanggal = test_data.index

                # Tampilkan Ringkasan
                display(widgets.HTML(f"<h3>✅ Berhasil Menggabungkan {len(port_ticker)} Aset</h3>"))
                display(test_data.head())

    def on_reset_clicked(b):
        global uploader
        uploaded_files.clear()
        uploader = create_uploader()
        uploader_container.children = [uploader, process_button, reset_button]
        with out_result:
            clear_output()
            print("🔄 Berhasil direset.")

    # Init UI
    uploader = create_uploader()
    process_button = widgets.Button(description='Process Data', button_style='success', icon='check')
    process_button.on_click(on_process_clicked)
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    reset_button.on_click(on_reset_clicked)

    uploader_container.children = [uploader, process_button, reset_button]
    print("# QuantGenius File Management")
    display(widgets.VBox([uploader_container]))

# --- PANEL B: MONTE CARLO ---
def create_mc_panel():
    style = {'description_width': 'initial'}
    w_n_hari = widgets.IntSlider(value=252, min=20, max=5000, description='Hari Bursa:', style=style)
    w_mu = widgets.FloatSlider(value=0.12, min=-0.5, max=0.5, step=0.01, description='Drift (Mu):', style=style)
    w_sigma = widgets.FloatSlider(value=0.25, min=0.01, max=1.0, step=0.01, description='Volatility:', style=style)
    w_s0 = widgets.FloatText(value=100.0, description='Harga Awal:', style=style)
    btn_run = widgets.Button(description="Run Simulation", button_style='success', icon='play')

    def on_mc_click(b):
        global dataHarga, test_data, tanggal
        with out_result:
            clear_output(wait=True)
            n_saham, n_hari = 30, w_n_hari.value
            dt = 1/252
            S0 = np.full(n_saham, w_s0.value)
            # Menghapus seed agar selalu random
            returns = np.exp((w_mu.value - 0.5 * w_sigma.value**2) * dt +
                             w_sigma.value * np.sqrt(dt) * np.random.standard_normal((n_hari, n_saham)))
            path = np.ones((n_hari, n_saham))
            path[0] = S0
            for t in range(1, n_hari): path[t] = path[t-1] * returns[t]

            # df_sim = pd.DataFrame(path, columns=[f"SIM_{i+1:02d}" for i in range(n_saham)])
            # df_sim.index = pd.date_range(start='2024-01-01', periods=n_hari, freq='B').date
            # df_sim.index.name = 'Date'

            df_sim = pd.DataFrame(path, columns=[f"SIM_{i+1:02d}" for i in range(n_saham)])
            df_sim.index = pd.date_range(start='2024-01-01', periods=n_hari, freq='B')
            df_sim.index.name = 'Date'

            test_data = df_sim
            dataHarga = test_data.to_numpy()
            tanggal = test_data.index.to_numpy()

            print(test_data.head())

            out_chart, out_table = widgets.Output(), widgets.Output()
            with out_chart:
                df_sim.plot(figsize=(10, 4), alpha=0.6, legend=False)
                plt.title("Montecarlo Simulation Paths")
                plt.grid(True, alpha=0.3)
                plt.show()
            with out_table:
                display(df_sim.head(10).style.format("{:,.2f}"))

            tabs = widgets.Tab(children=[out_chart, out_table])
            tabs.set_title(0, '📈 Chart'); tabs.set_title(1, '📋 Data Preview')
            display(tabs)

    btn_run.on_click(on_mc_click)
    display(widgets.VBox([widgets.HTML("<h4>🎲 Monte Carlo Parameter</h4>"), w_n_hari, w_mu, w_sigma, w_s0, btn_run]))

# 3. Render Interface
def render_interface(change):
    with out_main:
        clear_output()
        with out_result: clear_output()
        if datasource_radio.value == 'csv':
            create_csv_panel()
        else:
            create_mc_panel()

datasource_radio.observe(render_interface, names='value')

# 4. Tampilan Utama
# print("🚀 QuantGenius Data Engine")
display(datasource_radio)
display(toggle_button)
# display(widgets.HTML("<hr style='border-top: 1px solid #ddd; margin: 0 0 0px 0;'>"))
display(out_main)
display(out_result)

# Start
render_interface({'new': 'csv'})

RadioButtons(description='<b>Select Data Source:</b>', options=(('Upload 30 CSV Files', 'csv'), ('Montecarlo S…

Output()

Output()

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Inisialisasi Widget
uploader = widgets.FileUpload(
    accept='',
    multiple=True,
    description='Pilih File'
)

output = widgets.Output()

def handle_upload(change):
    with output:
        clear_output()

        # Ambil daftar file dari widget
        uploaded_files = uploader.value

        if not uploaded_files:
            return

        # Hitung total ukuran (dalam Bytes)
        total_size = sum(f['size'] for f in uploaded_files)
        total_size_mb = total_size / (1024 * 1024)

        # BATAS MAKSIMAL (Sesuaikan dengan limit Colab)
        LIMIT_MB = 10.0

        if total_size_mb > LIMIT_MB:
            print("❌ TOTAL FILE TERLALU BESAR!")
            print(f"Ukuran terdeteksi: {total_size_mb:.2f} MB")
            print("-" * 40)
            print("⚠️ WARNING: Google Colab membatasi upload via widget maksimal 10MB.")
            print("SOLUSI:")
            print(f"1. Pilih maksimal 2-3 file saja agar total di bawah {LIMIT_MB}MB.")
            print("2. Klik tombol upload lagi untuk file sisanya (bertahap).")
            print("3. Atau gunakan panel 'Files' di folder kiri Colab untuk file besar.")
            print("-" * 40)

            # Reset widget agar user bisa mencoba lagi dengan jumlah file lebih sedikit
            # Catatan: Menghapus value uploader secara programatik
            uploader.value = ()
        else:
            print(f"✅ BERHASIL: {len(uploaded_files)} file siap diproses.")
            print(f"Total ukuran: {total_size_mb:.2f} MB")
            for f in uploaded_files:
                print(f"- {f['name']} ({f['size']/1024:.1f} KB)")

# Hubungkan fungsi dengan perubahan nilai pada widget
uploader.observe(handle_upload, names='value')

# Tampilkan ke layar
display(widgets.HTML("<b>Upload File (Maksimal Total 10MB per sesi):</b>"))
display(uploader, output)

HTML(value='<b>Upload File (Maksimal Total 10MB per sesi):</b>')

FileUpload(value={}, description='Pilih File', multiple=True)

Output()

In [1]:
from google.colab import files

uploaded = files.upload()

print("Upload selesai!")

for filename in uploaded.keys():
    print("File:", filename)

Saving ibm.us.txt to ibm.us.txt
Saving ge.us.txt to ge.us.txt
Saving hpq.us.txt to hpq.us.txt
Saving jnj.us.txt to jnj.us.txt
Saving mcd.us.txt to mcd.us.txt
Saving cnp.us.txt to cnp.us.txt
Saving dte.us.txt to dte.us.txt
Saving cvx.us.txt to cvx.us.txt
Saving jpm.us.txt to jpm.us.txt
Saving cat.us.txt to cat.us.txt
Saving ko.us.txt to ko.us.txt
Saving ed.us.txt to ed.us.txt
Saving aa.us.txt to aa.us.txt
Saving c.us.txt to c.us.txt
Saving axp.us.txt to axp.us.txt
Saving ip.us.txt to ip.us.txt
Saving bmy.us.txt to bmy.us.txt
Saving lmt.us.txt to lmt.us.txt
Saving kr.us.txt to kr.us.txt
Saving cl.us.txt to cl.us.txt
Saving gd.us.txt to gd.us.txt
Saving f.us.txt to f.us.txt
Saving dis.us.txt to dis.us.txt
Saving ba.us.txt to ba.us.txt
Saving fdx.us.txt to fdx.us.txt
Saving eix.us.txt to eix.us.txt
Saving hal.us.txt to hal.us.txt
Saving hd.us.txt to hd.us.txt
Saving dd.us.txt to dd.us.txt
Saving lly.us.txt to lly.us.txt
Upload selesai!
File: ibm.us.txt
File: ge.us.txt
File: hpq.us.txt
File

In [ ]:
import aiohttp
import asyncio
import urllib3
import base64
import numpy as np
http = urllib3.PoolManager()
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from IPython.display import display, clear_output, HTML, Javascript

def calculate_metrics(equity_array, initial_equity_value, risk_free_rate=0.02):
    """Menghitung Total Return, Max Drawdown, Sharpe Ratio, dan Sortino Ratio."""

    if len(equity_array) == 0:
        return {"Total Return": np.nan, "Max Drawdown": np.nan, "Sharpe Ratio": np.nan, "Sortino Ratio": np.nan}

    total_return = (equity_array[-1] - initial_equity_value) / initial_equity_value
    daily_returns = pd.Series(equity_array).pct_change().dropna().values

    # Max Drawdown
    cumulative_return = np.insert(equity_array / initial_equity_value, 0, 1.0)
    peak = np.maximum.accumulate(cumulative_return)
    drawdown = (peak - cumulative_return) / peak
    max_drawdown = np.max(drawdown)

    if len(daily_returns) < 1:
        return {"Total Return": total_return, "Max Drawdown": max_drawdown, "Sharpe Ratio": np.nan, "Sortino Ratio": np.nan}

    annual_risk_free_rate = risk_free_rate / 252
    avg_daily_return = np.mean(daily_returns)
    std_daily_return = np.std(daily_returns)

    # Sharpe Ratio
    if std_daily_return == 0:
        sharpe_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
    else:
        sharpe_ratio = (avg_daily_return - annual_risk_free_rate) / std_daily_return * np.sqrt(252)

    # Sortino Ratio
    downside_returns = daily_returns[daily_returns < 0]
    if len(downside_returns) == 0:
        sortino_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
    else:
        downside_deviation = np.std(downside_returns)
        if downside_deviation == 0:
            sortino_ratio = np.inf if avg_daily_return > annual_risk_free_rate else np.nan
        else:
            sortino_ratio = (avg_daily_return - annual_risk_free_rate) / downside_deviation * np.sqrt(252)

    return {"Total Return": total_return, "Max Drawdown": max_drawdown, "Sharpe Ratio": sharpe_ratio, "Sortino Ratio": sortino_ratio}

def display_full_report_new_tab(fig, daftar_saham, df_perf):
    # 1. Konversi Grafik Plotly ke HTML (hanya bagian div dan script)
    # Kita tidak menggunakan full_html=True agar bisa kita bungkus sendiri
    plot_div = fig.to_html(full_html=False, include_plotlyjs='cdn')

    # HTML Daftar Saham (Scrollbox)
    saham_items = "".join([f'<div style="background:white; padding:8px; border:1px solid #eee; border-radius:4px; font-size:12px;"><b>{s.split(" - ")[0]}</b> - {s.split(" - ")[1]} - {s.split(" - ")[2]}</div>' for s in daftar_saham])

    # 4. Gabungkan Semuanya menjadi satu Dokumen HTML utuh
    full_document = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Quantxi System Report</title>
        <style>
            body {{ background-color: #f4f7f6; padding: 40px; font-family: sans-serif; }}
            .container {{ max-width: 1200px; margin: auto; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div style="width: 1200px; background:#fff; padding:20px; border-radius:10px; margin-bottom:20px; border: 1px solid #ddd;">
                <h3 style="margin:0; font-family:sans-serif; color:#2c3e50;">Portfolio<span style="margin-left: 750px; font-family:sans-serif; color:#2c3e50">
                    Period of Data : 01/01/2021 - 01/01/2026 </span></h3>
                <div style="max-height: 150px; overflow-y: auto; padding: 10px; background: #f9f9f9; border-radius: 5px; display:grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap:10px; font-family:sans-serif;">
                    {saham_items}
                </div>
            </div>
            <div style="width: 1200px; background:white; padding:15px; border-radius:10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
                {plot_div}
            </div>
            <div style="width: 1200px; margin-top:30px; font-family:sans-serif;">
                <h2 style="color:#2c3e50;">Performance Comparison</h2>
                <style>
                    .perf-table {{ width:100%; border-collapse:collapse; margin-top:10px; background:white; }}
                    .perf-table th {{ background:#2c3e50; color:white; padding:12px; text-align:center; }}
                    .perf-table td {{ border:1px solid #dee2e6; padding:10px; text-align:center; }}
                    .perf-table tr:nth-child(even) {{ background:#f2f2f2; }}
                </style>
                {df_perf.to_html(index=False, classes='perf-table')}
            </div>
        </div>
    </body>
    </html>
    """

    # 5. Encode dan Kirim ke Jendela Baru via JavaScript
    full_html_encoded = base64.b64encode(full_document.encode()).decode()

    js_code = f"""
    <script>
        try {{
            var fullHTML = atob("{full_html_encoded}");
            var newWindow = window.open();
            if (newWindow) {{
                newWindow.document.open();
                newWindow.document.write(fullHTML);
                newWindow.document.title = "Laporan Lengkap Quantxi";
                newWindow.document.close();
            }} else {{
                alert("Popup diblokir! Harap izinkan popup untuk melihat laporan.");
            }}
        }} catch (error) {{
            console.error("Error:", error);
            alert("Gagal membuka laporan: " + error.message);
        }}
    </script>
    """
    display(HTML(js_code))

reset = http.request("DELETE", "https://www.quantgenius.ai/api/resetdata", headers = {"X-QUANTGENIUS-APIKEY":"2NV4yE070VhcarVhZRJN7u3ZWP8zeq9oCsely819"})
print(reset.json())

# total_iterations = 30
total_iterations = len(test_data)

# 2. Inisialisasi Variabel
stock_price = np.zeros(30, dtype='d')
stock_price_sebelum = np.zeros(30, dtype='d')
stock_position_size = np.zeros(30, dtype='d')

initial_stockPrice = np.array(dataHarga[0], dtype='d')
# initial_eq = 1000000 #initial_equity.value
# cash_balance = initial_eq

# GANTI np.array APPEND DENGAN LIST APPEND (LEBIH CEPAT)
quantxi_equity_list = []
buyhold_equity_list = []
market_value_list = []

previous_signal_hashcode = "0000000000000000000000000000000000000000000000000000000000000000"
dataIdx = 0
accrued_interest = 0


initial_equity = 1000000
min_excessLiquidity = initial_equity
cash_balance = initial_equity
commission = 0.001
spread = 0.001
interest_rate = 0.05
regT_margin_rate = 0.5
maintenance_margin_rate = 0.25

stock_price_sebelum = dataHarga[dataIdx]

# 3. Loop Simulasi
while dataIdx < total_iterations:

  stock_price = dataHarga[dataIdx]

  # Logika Bunga
  if cash_balance < 0:
    daily_Interest = cash_balance * (interest_rate / 360) * (pd.Timestamp(tanggal[dataIdx]) - pd.Timestamp(tanggal[dataIdx-1])).days
    accrued_interest += daily_Interest

  if pd.Timestamp(tanggal[dataIdx]).day == 1:
    cash_balance += accrued_interest
    accrued_interest = 0

  market_value = np.sum(stock_price * stock_position_size)
  equity_with_loanValue = cash_balance + market_value + accrued_interest

  portfolio_data = [{"price": stock_price[i], "position_size": stock_position_size[i]} for i in range(30)]
  data_input = {
    "prev_signal_hashcode": previous_signal_hashcode,
    "equity_balance": equity_with_loanValue,
    "portfolio_data": portfolio_data
  }

  response = http.request("POST", "https://www.quantgenius.ai/api/adddata", headers = {"X-QUANTGENIUS-APIKEY":"2NV4yE070VhcarVhZRJN7u3ZWP8zeq9oCsely819", "Content-Type": 'application/json'}, json = data_input)

  # signal_output = response.json()["data"]

  response_data = response.json()
  signal_output = response_data.get("data", {
    "signal_hashcode": previous_signal_hashcode,
    "trade_signal": [{"action": "HOLD", "quantity": 0} for _ in range(30)]
  })

  # previous_signal_hashcode = signal_output["signal_hashcode"]
  previous_signal_hashcode = signal_output.get("signal_hashcode", previous_signal_hashcode)

  # ----------------------------------------------------------------------------------
  # TRADE TRANSACTION (Dioptimasi dengan Vektorisasi NumPy)
  # ----------------------------------------------------------------------------------
  maintenance_margin_req = market_value * maintenance_margin_rate
  excess_liquidity = equity_with_loanValue - maintenance_margin_req
  regT_margin_req = market_value * regT_margin_rate
  excess_equity = equity_with_loanValue - regT_margin_req

  if excess_liquidity > 0:
    # Inisialisasi variabel
    estimate_imr = 0
    estimate_comm = 0
    filled_percentage = 0

    # Loop pertama untuk menghitung estimate_imr dan estimate_comm
    for i in range(30):
        action = signal_output["trade_signal"][i]["action"]
        quantity = float(signal_output["trade_signal"][i]["quantity"])
        price = stock_price[i]

        if action == "BUY":
            estimate_imr += (quantity * price) * regT_margin_rate
            estimate_comm += abs(quantity) * price * (commission + spread)
        elif action == "SELL":
            estimate_imr -= (quantity * price) * regT_margin_rate
            estimate_comm += abs(quantity) * price * (commission + spread)
        else:
            # Tidak melakukan apa-apa untuk action selain BUY/SELL
            pass

    # Menghitung filled_percentage
    total_estimate = estimate_imr + estimate_comm

    if total_estimate <= 0:
        filled_percentage = 1
    elif excess_equity <= 0:
        filled_percentage = 0
    elif excess_equity > total_estimate:
        filled_percentage = 1
    else:
        filled_percentage = excess_equity / total_estimate

    # Inisialisasi array
    filledOrder = []
    filledPrice = []
    tradeValue = []
    commission_arr = []

    # Loop kedua untuk menghitung order yang terisi
    for i in range(30):
        action = signal_output["trade_signal"][i]["action"]
        quantity = float(signal_output["trade_signal"][i]["quantity"])
        price = stock_price[i]

        if action == "BUY":
            filledOrder.append(quantity * filled_percentage)
            filledPrice.append(price)
        elif action == "SELL":
            filledOrder.append(-(quantity * filled_percentage))
            filledPrice.append(price)
        else:
            filledOrder.append(0)
            filledPrice.append(0)

        # Hitung trade value dan commission
        trade_value_item = filledOrder[i] * filledPrice[i]
        commission_item = abs(filledOrder[i]) * filledPrice[i] * (commission + spread)

        tradeValue.append(trade_value_item)
        commission_arr.append(commission_item)

    # Hitung total
    total_trade_value = sum(tradeValue)
    total_commission = sum(commission_arr)

    # Update Cash dan Posisi
    cash_balance -= total_trade_value + total_commission
    stock_position_size += filledOrder

  stock_price_sebelum = dataHarga[dataIdx]

  quantxi_equity_list.append(equity_with_loanValue)

  # Buy & Hold Equity (Vektorisasi Cepat)
  bh_position = initial_equity / 30 / initial_stockPrice
  buyhold_equity = np.sum(bh_position * stock_price)
  buyhold_equity_list.append(buyhold_equity)

  market_value_list.append(market_value)

  if excess_liquidity < min_excessLiquidity:
    min_excessLiquidity = excess_liquidity

  dataIdx += 1
# Konversi List ke Array NumPy
quantxi_equity_array = np.array(quantxi_equity_list, dtype='d')
buyhold_equity_array = np.array(buyhold_equity_list, dtype='d')
market_value_array = np.array(market_value_list, dtype='d')

# Hitung Metrik
metrics_qg = calculate_metrics(quantxi_equity_array, initial_equity)
metrics_bh = calculate_metrics(buyhold_equity_array, initial_equity)

# clear_output(wait=True) # Hapus progress bar final

# --- KONSTANTA ---
WINDOW = 252*10
ANNUAL_TRADING_DAYS = 252
RISK_FREE_RATE = 0.0 # Asumsi Rf harian = 0

# --- VISUALISASI GRAFIK ---
dates = pd.to_datetime(tanggal[:total_iterations])
qg_returns = (quantxi_equity_array - initial_equity) / initial_equity
bh_returns = (buyhold_equity_array - initial_equity) / initial_equity

# ----------------------------------------------------------------------------------
# --- PERHITUNGAN VOLATILITAS, RETURNS & RASIO ---
# ----------------------------------------------------------------------------------

# Log Returns Harian
qg_log_returns = pd.Series(np.diff(np.log(quantxi_equity_array)))
bh_log_returns = pd.Series(np.diff(np.log(buyhold_equity_array)))

# Volatilitas Bergulir Harian (StDev)
qg_rolling_stdev = qg_log_returns.rolling(window=WINDOW).std()
bh_rolling_stdev = bh_log_returns.rolling(window=WINDOW).std()

# 1. Rolling Annualized Volatility (untuk Plot 7)
qg_rolling_volatility = qg_rolling_stdev * np.sqrt(ANNUAL_TRADING_DAYS)
bh_rolling_volatility = bh_rolling_stdev * np.sqrt(ANNUAL_TRADING_DAYS)

# 2. Rolling Annualized Mean Return
qg_rolling_mean_ret = qg_log_returns.rolling(window=WINDOW).mean() * ANNUAL_TRADING_DAYS
bh_rolling_mean_ret = bh_log_returns.rolling(window=WINDOW).mean() * ANNUAL_TRADING_DAYS

# 3. Rolling Sharpe Ratio (untuk Plot 5)
qg_rolling_sharpe = (qg_rolling_mean_ret - RISK_FREE_RATE) / qg_rolling_volatility
bh_rolling_sharpe = (bh_rolling_mean_ret - RISK_FREE_RATE) / bh_rolling_volatility

qg_rolling_sharpe = qg_rolling_sharpe.dropna()
bh_rolling_sharpe = bh_rolling_sharpe.dropna()

print(((qg_log_returns.mean() * ANNUAL_TRADING_DAYS) - RISK_FREE_RATE) / (qg_log_returns.std() * np.sqrt(ANNUAL_TRADING_DAYS)))
print(((bh_log_returns.mean() * ANNUAL_TRADING_DAYS) - RISK_FREE_RATE) / (bh_log_returns.std() * np.sqrt(ANNUAL_TRADING_DAYS)))

# 4. Rolling Sortino Ratio (untuk Plot 6)
qg_downside_returns = qg_log_returns[qg_log_returns < 0]
bh_downside_returns = bh_log_returns[bh_log_returns < 0]
qg_rolling_downside_dev = qg_downside_returns.rolling(window=WINDOW).std()
bh_rolling_downside_dev = bh_downside_returns.rolling(window=WINDOW).std()
qg_annualized_downside_dev = qg_rolling_downside_dev * np.sqrt(ANNUAL_TRADING_DAYS)
bh_annualized_downside_dev = bh_rolling_downside_dev * np.sqrt(ANNUAL_TRADING_DAYS)
qg_rolling_sortino = qg_rolling_mean_ret / qg_annualized_downside_dev
bh_rolling_sortino = bh_rolling_mean_ret / bh_annualized_downside_dev

# Rasio Keuangan
epsilon = 1000000
# Menggunakan market_value_safe untuk menghindari pembagian nol yang menyebabkan NaN pada Plot 3 & 4
market_value_safe = np.where(market_value_array == 0, epsilon, market_value_array)

qg_equity_to_market_value = quantxi_equity_array / market_value_safe
qg_leverage = market_value_safe / quantxi_equity_array # Menggunakan safe MV juga di sini

dates_for_returns = dates[1:]
dates_for_volatility = dates[WINDOW:]

# --- 1. Data Preparation (Equity & Risk) ---
quantxi_equity_df = pd.DataFrame(quantxi_equity_array, columns=['equity_value'])
buyhold_equity_df = pd.DataFrame(buyhold_equity_array, columns=['buyhold_value'])
market_value_df = pd.DataFrame(market_value_array, columns=['market_value'])

# Penanganan pembagian nol
market_val_safe = market_value_df['market_value'].replace(0, 1)
ratio_solvability = quantxi_equity_df['equity_value'].values / market_val_safe.values

# Drawdown Calculation
qg_cumulative_return = quantxi_equity_array / initial_equity
qg_peak = np.maximum.accumulate(qg_cumulative_return)
qg_drawdown_plot = (qg_peak - qg_cumulative_return) / qg_peak

bh_cumulative_return = buyhold_equity_array / initial_equity
bh_peak = np.maximum.accumulate(bh_cumulative_return)
bh_drawdown_plot = (bh_peak - bh_cumulative_return) / bh_peak

# Sinkronisasi Sumbu Row 2
qg_plot_sync = 1 - qg_drawdown_plot
bh_plot_sync = 1 - bh_drawdown_plot

# Contoh Daftar Saham dengan format lengkap
daftar_saham = [
    f"{s} - IDX - Indonesia" for s in [
        'BBCA-Bank Central Asia', 'BBRI-Bank Rakyat Indoensia', 'BMRI-Bank Mandiri', 'TLKM-PT. Telkom', 'ASII-PT. Astra Indonesia', 'GOTO-PT. Gojek Indonesia', 'UNVR', 'ICBP', 'BBNI', 'ADRO',
        'PGAS', 'UNTR', 'KLBF', 'CPIN', 'BRPT', 'AMRT', 'INKP', 'TPIA', 'MDKA', 'ANTM',
        'ITMG', 'PTBA', 'SMGR', 'GGRM', 'HMSP', 'EXCL', 'ISAT', 'BUKA', 'HRUM', 'AKRA'
    ]
]

# --- 3. Create Subplots (3 Rows) ---
fig = make_subplots(
rows=3, cols=1,
subplot_titles=(
  '<b>Perbandingan Equity Curve: Quantxi vs Buy & Hold</b>',
  '<b>Anti Fragile (Solvability Ratio vs Drawdown)</b>',
  '<b>Rolling Sharpe Ratio Analysis</b>'
),
vertical_spacing=0.07,
specs=[[{"secondary_y": False}], [{"secondary_y": True}], [{"secondary_y": False}]]
)

# Row 1: Equity
fig.add_trace(go.Scatter(y=quantxi_equity_df['equity_value'], name='Equity Quantxi', line=dict(color='blue'), legend='legend1'), row=1, col=1)
fig.add_trace(go.Scatter(y=buyhold_equity_df['buyhold_value'], name='Buy & Hold', line=dict(color='red'), legend='legend1'), row=1, col=1)

# Row 2: Anti Fragile
fig.add_trace(go.Scatter(y=[1.0]*len(ratio_solvability), line=dict(width=0), showlegend=False, hoverinfo='skip'), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(y=ratio_solvability, name='DD Ratio (Q/BH)', line=dict(color='orange', width=3), fill='tonexty', fillcolor='rgba(255, 165, 0, 0.2)', legend='legend2'), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(y=qg_plot_sync, name='DD Quantxi', line=dict(color='green', width=1.5), legend='legend2'), row=2, col=1, secondary_y=True)
fig.add_trace(go.Scatter(y=bh_plot_sync, name='DD Buy & Hold', line=dict(color='yellow', width=1.5), legend='legend2'), row=2, col=1, secondary_y=True)

# Row 3: Sharpe Ratio
fig.add_trace(go.Scatter(y=qg_rolling_sharpe, name='QG Rolling Sharpe', line=dict(color='purple'), legend='legend3'), row=3, col=1)
fig.add_trace(go.Scatter(y=bh_rolling_sharpe, name='BH Rolling Sharpe', line=dict(color='green'), legend='legend3'), row=3, col=1)

# Konfigurasi Sumbu Row 2
fig.update_yaxes(range=[0, 1], tickvals=[0, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0], showgrid=True, gridcolor='LightGrey', row=2, col=1, secondary_y=False)
fig.update_yaxes(range=[0, 1], tickvals=[0, 0.2, 0.4, 0.6, 0.7, 0.8, 1.0], ticktext=["100%", "80%", "60%", "40%", "30%", "20%", "0%"], showgrid=False, row=2, col=1, secondary_y=True)
fig.add_hline(y=0.3, line_dash="dot", line_color="red", line_width=2, row=2, col=1)

# --- 4. Performance Comparison Table ---
# Kalkulasi data akhir
final_qg_return = ((quantxi_equity_array[-1] / initial_equity) - 1) * 100
final_bh_return = ((buyhold_equity_array[-1] / initial_equity) - 1) * 100
max_drawdown_qg = np.max(qg_drawdown_plot) * 100
max_drawdown_bh = np.max(bh_drawdown_plot) * 100

data_perf = {
"Metrik Performa": ["Final Return (%)", "Max Drawdown (%)", "Final Equity Value", "Sharpe Ratio (Avg)"],
"Sistem AI (Quantxi)": [f"{final_qg_return:.2f}%", f"{max_drawdown_qg:.2f}%", f"{quantxi_equity_array[-1]:,.0f}", f"{np.mean(qg_rolling_sharpe):.2f}"],
"Buy & Hold": [f"{final_bh_return:.2f}%", f"{max_drawdown_bh:.2f}%", f"{buyhold_equity_array[-1]:,.0f}", f"{np.mean(bh_rolling_sharpe):.2f}"]
}
df_perf = pd.DataFrame(data_perf)

# --- 5. Display Everything ---
fig.update_layout(width=1200, height=1400, margin=dict(l=100, r=100, t=80, b=80))

# Menampilkan berurutan
display_full_report_new_tab(fig, daftar_saham, df_perf)

{'status': 'success', 'message': 'Your data has been successfully reset'}
0.5175165662392104
0.5443440164654806


## **4. Run Trade Simulation**
Menjalankan simulasi perdagangan secara sekuensial. Setiap baris data akan dikirim ke API **QuantGenius** untuk mendapatkan sinyal keputusan secara real-time.
> **Note:** Proses ini mensimulasikan kondisi pasar asli di mana data masa depan tidak tersedia bagi sistem AI (*No Look-Ahead Bias*).

In [ ]:
# @title ⚙️ Run Simulation { display-mode: "form" }
from tqdm.notebook import tqdm
import time


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. GENERASI DATA DUMMY (500 Hari)
np.random.seed(88)
n_days = 500
dates = pd.date_range(start='2024-01-01', periods=n_days, freq='B')

# Simulasi Equity Curve (Dimulai dari 100,000,000)
initial_capital = 100_000_000
bh_returns = np.random.normal(0.0005, 0.012, n_days) # Benchmark lebih volatil
qg_returns = np.random.normal(0.0008, 0.008, n_days) # QuantGenius lebih stabil (low vol)

equity_bh = initial_capital * (1 + bh_returns).cumprod()
equity_qg = initial_capital * (1 + qg_returns).cumprod()

df_perf = pd.DataFrame({
    'Buy_and_Hold': equity_bh,
    'QuantGenius': equity_qg
}, index=dates)

# 2. FUNGSI ANALISIS PERFORMA
def calculate_metrics(series):
    returns = series.pct_change().dropna()
    total_return = (series.iloc[-1] / series.iloc[0]) - 1
    cagr = (series.iloc[-1] / series.iloc[0])**(252/len(series)) - 1
    vol = returns.std() * np.sqrt(252)
    sharpe = (cagr - 0.03) / vol # Risk-free rate diasumsikan 3%

    # Max Drawdown
    rolling_max = series.cummax()
    drawdown = (series - rolling_max) / rolling_max
    max_dd = drawdown.min()

    return {
        "Total Return": f"{total_return*100:.2f}%",
        "CAGR": f"{cagr*100:.2f}%",
        "Ann. Volatility": f"{vol*100:.2f}%",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "Max Drawdown": f"{max_dd*100:.2f}%"
    }

# 3. RENDER DASHBOARD
out_perf = widgets.Output()

def show_performance_dashboard():
    with out_perf:
        clear_output()

        # --- TAB 1: COMPARISON CHART ---
        out_chart = widgets.Output()
        with out_chart:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [3, 1]})

            # Plot Equity Curve
            ax1.plot(df_perf['Buy_and_Hold'], label='Buy & Hold', color='#999999', alpha=0.7)
            ax1.plot(df_perf['QuantGenius'], label='QuantGenius AI', color='#1a73e8', linewidth=2)
            ax1.set_title("Equity Curve Comparison", fontsize=14, fontweight='bold')
            ax1.set_ylabel("Portfolio Value")
            ax1.legend()
            ax1.grid(True, alpha=0.3)

            # Plot Drawdown
            dd_bh = (df_perf['Buy_and_Hold'] / df_perf['Buy_and_Hold'].cummax()) - 1
            dd_qg = (df_perf['QuantGenius'] / df_perf['QuantGenius'].cummax()) - 1
            ax2.fill_between(dates, dd_bh, 0, color='#999999', alpha=0.2, label='B&H DD')
            ax2.fill_between(dates, dd_qg, 0, color='#1a73e8', alpha=0.3, label='QG DD')
            ax2.set_ylabel("Drawdown")
            ax2.set_title("Underwater Chart (Drawdown Analysis)", fontsize=10)

            plt.tight_layout()
            plt.show()

        # --- TAB 2: PERFORMANCE REPORT ---
        out_report = widgets.Output()
        with out_report:
            qg_m = calculate_metrics(df_perf['QuantGenius'])
            bh_m = calculate_metrics(df_perf['Buy_and_Hold'])

            report_df = pd.DataFrame([bh_m, qg_m], index=['Buy & Hold', 'QuantGenius']).T

            display(widgets.HTML("<h3>📊 Performance Strategy Report</h3>"))
            styled_report = report_df.style.set_properties(**{
                'text-align': 'center', 'border': '1px solid #eee', 'min-width': '150px'
            }).set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#1a73e8')]},
                {'selector': 'td:nth-child(3)', 'props': [('font-weight', 'bold'), ('background-color', '#e8f0fe')]}
            ])
            display(styled_report)

        # --- TAB 3: MONTHLY RETURNS HEATMAP (Optional but Pro) ---
        out_monthly = widgets.Output()
        with out_monthly:
            display(widgets.HTML("<h4>📅 Monthly Returns Breakdown</h4>"))
            print("Feature ini bisa ditambahkan dengan library seaborn untuk melihat performa per bulan.")
            display(df_perf.resample('M').last().pct_change().tail(10).style.format("{:.2%}"))

        # ORGANIZE TABS
        tabs = widgets.Tab(children=[out_chart, out_report, out_monthly])
        tabs.set_title(0, '📈 Equity & Drawdown')
        tabs.set_title(1, '📋 Performance Metrics')
        tabs.set_title(2, '📅 Periodical Analysis')
        display(tabs)

show_performance_dashboard()
display(out_perf)

Output()